# Verify Token and BIO Tag Count Match

This notebook verifies that the number of tokens in each row matches the number of BIO tags.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv('label_token_data_with_BIO_tags_v2.csv', header=None)
print(f"Total rows in dataset: {len(df)}")
df.head()

Total rows in dataset: 1995


,0,1,2,3
0,sentence,label,tokens,bio_tags
1,I want gift suggestion for general in all gifts,2,want|gift|suggestion|for|general|in|all|gifts,O|B|O|O|O|O|O|B
2,I want gift suggestion for wife in all gifts,2,want|gift|suggestion|for|wife|in|all|gifts,O|B|O|O|O|O|O|B
3,Hi I need to send a birthday scroll to my youn...,2,hi|need|to|send|birthday|scroll|to|my|younger|...,O|O|O|O|B|I|O|O|O|O|O|O|O|O
4,Birthday,2,birthday,O


In [3]:
# Function to count tokens and tags
def count_elements(row):
    tokens = row[2].split('|') if pd.notna(row[2]) else []
    tags = row[3].split('|') if pd.notna(row[3]) else []
    return len(tokens), len(tags)

# Apply the function to each row
df[['token_count', 'tag_count']] = df.apply(lambda row: pd.Series(count_elements(row)), axis=1)

In [4]:
# Check for mismatches
df['mismatch'] = df['token_count'] != df['tag_count']

# Count violations
violation_count = df['mismatch'].sum()
print(f"\n{'='*60}")
print(f"Number of rows with token-tag mismatch: {violation_count}")
print(f"Percentage of violations: {(violation_count/len(df)*100):.2f}%")
print(f"{'='*60}\n")


Number of rows with token-tag mismatch: 217
Percentage of violations: 10.88%



In [5]:
# Display rows with mismatches
if violation_count > 0:
    print(f"Rows with mismatches (showing first 20):")
    mismatch_rows = df[df['mismatch']].copy()
    mismatch_rows['row_index'] = mismatch_rows.index
    display_cols = ['row_index', 0, 'token_count', 'tag_count']
    print(mismatch_rows[display_cols].head(20).to_string(index=False))
else:
    print("✓ All rows have matching token and tag counts!")

Rows with mismatches (showing first 20):
 row_index                                                                                                                                                                                                                                                                                                                    0  token_count  tag_count
        27                                                                                                                                                                                                                                                                                      I want gift suggestion for wife            5          4
        36                                                                                                                                                                                                                                                                     

In [6]:
# Detailed view of specific mismatched rows
if violation_count > 0:
    print("\nDetailed view of first 5 mismatched rows:\n")
    for idx in mismatch_rows.head(5).index:
        row = df.loc[idx]
        print(f"Row {idx}:")
        print(f"  Text: {row[0][:80]}..." if len(str(row[0])) > 80 else f"  Text: {row[0]}")
        print(f"  Tokens ({row['token_count']}): {row[2][:100]}..." if len(str(row[2])) > 100 else f"  Tokens ({row['token_count']}): {row[2]}")
        print(f"  Tags ({row['tag_count']}): {row[3][:100]}..." if len(str(row[3])) > 100 else f"  Tags ({row['tag_count']}): {row[3]}")
        print(f"  Difference: {row['token_count'] - row['tag_count']}")
        print()


Detailed view of first 5 mismatched rows:

Row 27:
  Text: I want gift suggestion for wife
  Tokens (5): want|gift|suggestion|for|wife
  Tags (4): O|B|O|O
  Difference: 1

Row 36:
  Text: I want a gift for wife for Karva Chauth
  Tokens (7): want|gift|for|wife|for|karva|chauth
  Tags (6): O|B|O|O|O|O
  Difference: 1

Row 93:
  Text: I want gift suggestion for wife
  Tokens (5): want|gift|suggestion|for|wife
  Tags (4): O|B|O|O
  Difference: 1

Row 127:
  Text: I want to gift mu mom on mother's day under what is suitable to buy???
  Tokens (14): want|to|gift|mu|mom|on|mother|day|under|what|is|suitable|to|buy
  Tags (13): O|O|O|O|O|O|O|O|O|O|O|O|O
  Difference: 1

Row 155:
  Text: I want gift suggestion for wife
  Tokens (5): want|gift|suggestion|for|wife
  Tags (4): O|B|O|O
  Difference: 1



In [7]:
# Summary statistics
print("\nSummary Statistics:")
print(f"Min token count: {df['token_count'].min()}")
print(f"Max token count: {df['token_count'].max()}")
print(f"Mean token count: {df['token_count'].mean():.2f}")
print(f"\nMin tag count: {df['tag_count'].min()}")
print(f"Max tag count: {df['tag_count'].max()}")
print(f"Mean tag count: {df['tag_count'].mean():.2f}")


Summary Statistics:
Min token count: 1
Max token count: 105
Mean token count: 12.85

Min tag count: 1
Max tag count: 101
Mean tag count: 12.70
